# Acervo que Fala — Notebook 04 (v6): o sistema redesenhado

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

Este notebook testa uma hipótese levantada pelo autor antes do julgamento do bake-off: **parte dos erros não vinha do modelo, e sim de instruções mal desenhadas**. A análise confirmou três mecanismos (documentados em `avaliacao/analise_prompt_rubrica.md` no repositório): frases de exemplo do prompt copiadas literalmente para a saída ("sobre a argila bege" apareceu numa bolsa de fio de tucum), perguntas que induzem resposta na observação ("incluindo bordas, faixas e acabamentos"), e palavras-gatilho lidas sem a negação ("não há close-up" → alt marcado como detalhe).

**O que a v6 muda — o sistema, não o modelo (mesmo Qwen3-VL-8B):**

- **Observação v3 em seções nomeadas** (OBJETO, MATERIAIS E CORES, PADRÕES…, ENQUADRAMENTO, ARTEFATOS), com contexto do acervo e uma guarda explícita: reconhecer materiais, nunca adivinhar significado;
- **Redação v9 com Contrato de Fontes**: sem fonte → não escreve (omitir é sempre permitido); incerteza herda-se da seção LEGIBILIDADE; divergência vira flag, nunca harmonização; exemplos só com lacunas [assim];
- **Rubrica v1.2 enxuta**: só o que o RAG realmente entrega (categoria + glossário) — as regras universais vivem no prompt;
- **Garantias em código**: enquadramento parseado da observação e injetado (fim do "Detalhe" indevido), cada artefato observado vira flag automaticamente (recall garantido), e pós-processamento remove "sobre fundo [cor]" residual.

O resultado se compara com o da v5 (mesmo modelo, sistema antigo): se a v6 limpar os defeitos, a hipótese do autor estava certa.

*Metodologia: projeto construído com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~35–45 min**. Deixe a aba aberta durante a execução.

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 20 objetos, com salvaguardas de imagem (~3 min)

O lote: os **5 objetos do smoke test** (para comparar com os notebooks anteriores) + **15 novos**, sorteados com seed fixa (reproduzível) entre os itens que **não** estão no conjunto de avaliação — o lote serve para testar o pipeline em escala, sem "viciar" nos itens que depois vão dar a nota.

Duas salvaguardas novas ao baixar cada foto:

- **Orientação EXIF**: fotos de câmera guardam a rotação numa etiqueta interna que os navegadores aplicam, mas o Python não — sem esta linha, o modelo poderia receber uma foto deitada sem ninguém saber. `ImageOps.exif_transpose` aplica a rotação correta.
- **Conversão para RGB**: garante que qualquer foto (escala de cinza, outros formatos de cor) chegue ao modelo no formato esperado.

Desta vez o registro completo de cada objeto (povo, materiais, dimensões, descrição curatorial...) **viaja junto** — a correção do bug do Notebook 03.

In [ ]:
import io, re, requests
from PIL import Image, ImageOps

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
IDS_SMOKE = [9196, 665, 51023, 63283, 78838]
# 15 novos: sorteio seed 42, estratificado por categoria, excluindo os 50 casos
# de avaliação (seleção documentada no repositório, commit da E7)
IDS_LOTE = [1376, 84811, 883523, 2081, 5011, 200648, 210680, 5146, 500179, 3411, 1366, 4156, 205095, 905, 500322]

CAMPOS_REGISTRO = ["Nome do item", "Povo", "Categoria", "Matéria-prima",
                   "Técnica de confecção", "Dimensões", "Função",
                   "Estado de origem", "Ano de aquisição do objeto", "Descrição"]

objetos = []
for item_id in IDS_SMOKE + IDS_LOTE:
    item = requests.get(f"{BASE}/items/{item_id}", timeout=60).json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))
    foto = ImageOps.exif_transpose(foto).convert("RGB")  # salvaguardas
    meta_bruto = requests.get(f"{BASE}/item/{item_id}/metadata", timeout=60).json()
    todos = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    registro = {c: todos.get(c, "") for c in CAMPOS_REGISTRO}
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto, "registro": registro})
    print(f"✓ {item_id} — {item['title']} ({registro['Povo']})")
print(f"{len(objetos)} objetos carregados")

In [ ]:
# Etapa 3 — Drive (rubrica v1.2) + embeddings do RAG
import json, os
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
# v1.2: rubrica enxuta — só categoria + glossário (o que a recuperação entrega);
# as regras universais moram no prompt de redação. Versões anteriores preservadas no Drive.
with open(f"{PROJETO}/dados/rubrica_v1_2.json", encoding="utf-8") as f:
    rubrica = json.load(f)
trechos = rubrica["trechos"]

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
vetores = embedder.encode([t["texto"] for t in trechos], convert_to_tensor=True)

def recuperar(consulta, k=3):
    v = embedder.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(v, vetores)[0]
    return [trechos[i] for i in scores.argsort(descending=True)[:k].tolist()]

print(f"rubrica {rubrica['versao']}: {len(trechos)} trechos indexados ✓")

In [ ]:
# Etapa 4 — Modelo (Qwen3-VL-8B em 4-bit, como nos notebooks anteriores)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("modelo carregado ✓")

## Etapa 5 — Observação v3: seções nomeadas (~15 min)

A observação deixou de ser texto corrido e virou um **formulário de seções** — cada uma responde uma pergunta neutra ("somente se existirem"), sem induzir resposta. As duas últimas seções (`ENQUADRAMENTO:` e `ARTEFATOS:`) são **consumidas pelo código**: o enquadramento é injetado na redação como decisão pronta, e cada artefato vira flag automaticamente. O contexto do acervo entra com uma guarda explícita — serve para reconhecer materiais, nunca para adivinhar o que o objeto é.

Depois do laço, a célula faz o parse das seções e prepara a versão da observação que a redação vai receber (sem as linhas que o código já consumiu).

In [ ]:
PROMPT_OBSERVACAO_V3 = """Você está diante da fotografia de um objeto do acervo de um museu — objetos etnográficos de povos indígenas do Brasil, fotografados em estúdio. Este contexto serve para você reconhecer materiais e situações de estúdio; NÃO use para adivinhar o que o objeto é ou significa. Descreva somente o que está visível NESTA fotografia.

Preencha as seções abaixo, nesta ordem:

OBJETO: o que se vê, em uma frase — forma geral, sem nomear função nem significado.
MATERIAIS E CORES: os materiais aparentes e suas cores, do maior para o menor. Material que não dá para identificar recebe o termo genérico ("fibra", "madeira clara") — nunca chute espécie ou origem.
PADRÕES E TEXTURAS: desenhos, tramas e acabamentos visíveis, descritos pela forma (linhas, xadrez, diagonais) — somente se existirem.
PARTES E QUANTIDADES: partes distinguíveis e contáveis (tubos, furos, alças, penas destacadas).
POSIÇÃO: como o objeto está na foto (de pé, deitado, inclinado) e partes internas visíveis (boca, interior, verso).
LEGIBILIDADE: o que estiver ilegível ou incerto — declare a incerteza em vez de estimar.
FUNDO E ESTÚDIO: o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).
ENQUADRAMENTO: inteiro OU detalhe — "detalhe" SÓ se a foto mostra claramente apenas parte do objeto; objeto que encosta ou sangra nas margens conta como inteiro.
ARTEFATOS: os artefatos de estúdio vistos, separados por vírgula, ou "nenhum".

Regra geral: o que não está visível não existe para esta descrição. Responda em português."""

for n, obj in enumerate(objetos, 1):
    obj["observacao"] = gerar(
        [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V3}],
        max_tokens=500,
    )
    print(f"[{n}/{len(objetos)}] {obj['titulo']} observado ✓")

# --- Parse das seções (o código consome ENQUADRAMENTO e ARTEFATOS) ---
CABECALHO = r"(?:OBJETO|MATERIAIS E CORES|PADRÕES E TEXTURAS|PARTES E QUANTIDADES|POSIÇÃO|LEGIBILIDADE|FUNDO E ESTÚDIO|ENQUADRAMENTO|ARTEFATOS)"

def secao(texto, nome):
    m = re.search(rf"{nome}\s*:\s*(.+?)(?=\n\s*\**{CABECALHO}\**\s*:|\Z)", texto, re.S | re.I)
    return m.group(1).strip().strip("*").strip() if m else ""

for obj in objetos:
    obs = obj["observacao"]
    enq = secao(obs, "ENQUADRAMENTO").lower()
    obj["enquadramento"] = "detalhe" if enq.startswith("detalhe") else "inteiro"
    obj["enquadramento_ok"] = enq.startswith(("inteiro", "detalhe"))
    art = secao(obs, "ARTEFATOS")
    obj["artefatos_obs"] = [] if (not art or art.lower().startswith("nenhum")) else [
        a.strip(" .") for a in art.split(",") if a.strip(" .")
    ]
    # a redação recebe a observação SEM as duas linhas já consumidas pelo código
    obj["observacao_para_redacao"] = re.sub(
        rf"\n?\s*\**(ENQUADRAMENTO|ARTEFATOS)\**\s*:.*?(?=\n\s*\**{CABECALHO}\**\s*:|\Z)",
        "", obs, flags=re.S | re.I,
    ).strip()
    # consulta do RAG: identidade do item + o que a foto mostra (seções OBJETO e PADRÕES)
    obj["consulta_rag"] = (f"{obj['titulo']} ({obj['registro']['Categoria']}). "
                          f"{secao(obs, 'OBJETO')} {secao(obs, 'PADRÕES E TEXTURAS')}")[:400]

ok = sum(1 for o in objetos if o["enquadramento_ok"])
print(f"\nparse: {ok}/{len(objetos)} com ENQUADRAMENTO válido | "
      f"{sum(len(o['artefatos_obs']) for o in objetos)} artefatos observados no lote")

## Etapa 6 — Redação v9: Contrato de Fontes (~20 min)

O prompt foi redesenhado da estrutura, não por acréscimo (análise em `avaliacao/analise_prompt_rubrica.md`):

- cada insumo declara **o que autoriza** (observação → aparência; registro → fatos com atribuição; diretrizes → vocabulário);
- o **Contrato de Fontes** fica na posição de maior prioridade (depois dos dados, antes das saídas): sem fonte → não escreva; incerteza herda-se da seção LEGIBILIDADE; divergência vira flag, nunca harmonização; exemplos com lacunas [assim], nunca copiáveis;
- as saídas viraram **checklist positivo** (o que um bom texto contém), com só as proibições irredutíveis;
- o enquadramento chega **decidido** (variável injetada pelo código) e as flags de artefato são **automáticas** — o modelo se concentra em divergências e metadados suspeitos.

Depois da geração, o código aplica o pós-processamento determinístico no alt (remoção de "sobre fundo [cor]" residual) e funde as flags automáticas.

In [ ]:
PROMPT_REDACAO_V9 = """Você escreve descrições de acessibilidade para o acervo digital de um museu. Elas serão OUVIDAS por pessoas cegas, através de leitores de tela — escreva em linguagem cotidiana, com frases que funcionam no ouvido (ordem direta, sem parênteses longos), sem jargão de catálogo.

INSUMOS — cada um autoriza um tipo de informação:

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (autoriza: aparência — o que é visível):
{observacao}

ENQUADRAMENTO DECIDIDO NA OBSERVAÇÃO: {enquadramento}

REGISTRO DO MUSEU (autoriza: fatos — sempre com atribuição; o título nomeia o objeto):
{registro}

DIRETRIZES PARA ESTE TIPO DE OBJETO (autorizam: vocabulário e o que observar nesta categoria):
{diretrizes}

CONTRATO DE FONTES — prevalece sobre qualquer outra regra:
1. Cada informação escrita precisa de fonte: visível na observação, escrita no registro (e então leva atribuição) ou vocabulário das diretrizes. Sem fonte, não escreva — omitir é sempre permitido; um texto curto e todo verificável vale mais que um completo com um palpite.
2. Incerteza se herda: o que estiver na seção LEGIBILIDADE da observação, ou vier com hesitação ("parece", "talvez", "possivelmente"), sai do texto ou vira o termo genérico — nunca vira afirmação.
3. Divergência não se resolve no texto: quando observação e registro conflitam (nome, cor, quantidade, material, dimensão), não escolha um lado nem harmonize — registre em flags como divergencia_imagem_catalogo; no texto, o fato não visível fica com o registro e a aparência fica com a observação.
4. Os exemplos deste prompt mostram a FORMA das frases, com lacunas [assim]; preencha sempre com o conteúdo deste objeto, nunca com as palavras do exemplo.

PRODUZA TRÊS SAÍDAS:

A) alt_text — o que a fotografia mostra, para quem não a vê.
   Uma frase, no máximo 30 palavras, começando pelo objeto (nomeado pelo TÍTULO do registro) e pelo povo.
   Contém: o material, quando natural e sem tingimento; as cores, onde a cor informa (penas, miçangas, pinturas, tingimentos); a forma dos padrões (faixas, xadrez, losangos, geométrico); as quantidades da seção PARTES E QUANTIDADES da observação.
   As seções FUNDO E ESTÚDIO da observação nunca alimentam este texto nem o B — o fundo do estúdio não existe para as descrições. Quando a peça tem pintura ou decoração aplicada sobre a base, descreva nesta ordem: primeiro a decoração e suas cores, depois a base — "pintura [tipo] em [cores] sobre [material da base]". A base nunca aparece sozinha ("sobre a [material]") sem dizer o que está sobre ela.
   Se o ENQUADRAMENTO diz "detalhe", comece com "Detalhe de [objeto]"; se diz "inteiro", não mencione enquadramento nem orientação que não informa.
   Aves de penas: só as que o registro nomear, com a cor primeiro: "penas [cores] de [ave]".
   Não aparecem aqui: artefato de estúdio ou inventário, palavra de catálogo, medida.

B) descricao_objeto — o objeto em si, para quem quer conhecê-lo além da foto. Dois parágrafos:
   1º: abre direto com o objeto e sua função quando ela acrescenta algo (caça, ritual, preparo) — nunca o óbvio. A primeira informação vinda do registro leva a marca de atribuição: "segundo o registro do museu", "o registro informa que" ou equivalente — e TODO fato do catálogo que não é visível na foto (função, técnica, origem, ano, medidas, decorações que o registro descreve) carrega marca de atribuição na própria frase, com formulação variada. Depois, a aparência: formas, materiais e padrões em palavras comuns, cada informação dita uma vez.
   2º: os demais fatos do catálogo em frases naturais ("adquirido em [ano]"); a escala é a maior dimensão aproximada ("cerca de [número] cm de comprimento") — miniatura é dita miniatura; aves das penas detalhadas conforme o registro; significado cultural só se estiver no registro.
   Este texto descreve o objeto, não a fotografia: posição, fundo, enquadramento e a própria foto não existem aqui; relações que dependem do ponto de vista viram relações da peça ("decrescentes", "em degraus").
   O que não existe no objeto simplesmente não é mencionado — nada de "sem [coisa]" ou "não há [coisa]".

C) flags — o que precisa de revisão humana (os artefatos vistos na observação já serão registrados automaticamente; concentre-se no resto):
   - divergencia_imagem_catalogo: conflito entre o que a observação vê e o que o registro afirma — ou objeto visto diferente do que o título nomeia (use o título no texto e registre a diferença aqui);
   - metadado_suspeito: valor improvável no registro (dimensão absurda para o tipo de objeto, data impossível) ou contradição entre campos do próprio registro.
   Lista vazia [] se não houver nada.

Responda APENAS com JSON: {{"alt_text": "...", "descricao_objeto": "...", "flags": [{{"tipo": "...", "detalhe": "..."}}]}}"""

def pos_processar_alt(alt):
    # remoção determinística do fundo de estúdio residual ("sobre/em/contra [um|o] fundo ...")
    alt = re.sub(r",?\s*(sobre|em|contra)\s+(um\s+|o\s+)?fundo[^,\.]*", "", alt, flags=re.I)
    alt = re.sub(r",\s*,", ",", alt)
    alt = re.sub(r"\s{2,}", " ", alt).strip(" ,")
    if alt and not alt.endswith("."):
        alt += "."
    return alt

for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    achados = recuperar(obj["consulta_rag"])
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V9.format(
        observacao=obj["observacao_para_redacao"],
        enquadramento=obj["enquadramento"],
        registro=registro_txt,
        diretrizes="\n".join(f"- {t['texto']}" for t in achados),
    )
    resposta = gerar([{"type": "text", "text": prompt}], max_tokens=700)
    try:
        saida = extrair_json(resposta)
        obj["alt_text"] = pos_processar_alt(saida["alt_text"])
        obj["descricao_objeto"] = saida["descricao_objeto"]
        # flags automáticas dos artefatos observados + flags do modelo (sem duplicar artefatos)
        flags_auto = [{"tipo": "artefato_estudio", "detalhe": a} for a in obj["artefatos_obs"]]
        flags_modelo = [f for f in saida.get("flags", []) if f.get("tipo") != "artefato_estudio"]
        obj["flags"] = flags_auto + flags_modelo
        obj["json_valido"] = True
    except Exception:
        obj["alt_text"], obj["descricao_objeto"], obj["flags"] = resposta, "", []
        obj["json_valido"] = False
    print(f"[{n}/{len(objetos)}] {obj['titulo']}: {obj['alt_text'][:80]}... | flags: {len(obj['flags'])}")

## Etapa 7 — Verificação automática (com coerência de enquadramento)

Checagens herdadas: JSON válido; povo no alt; artefato nos textos; ≤30 palavras; atribuição; frases-etiqueta; "foi aquisição em"; ausências; foto no nível 2; especulação; frases vazias; "fundo" no alt (deve ser zero após o pós-processamento); qualidade das flags.

Novidades da v6: **coerência de enquadramento** (o alt começa com "Detalhe" se e somente se a observação decidiu "detalhe") e **parse da observação** (item sem a linha ENQUADRAMENTO válida conta como problema — a estrutura é parte do contrato).

In [ ]:
TERMOS_ARTEFATO = ["cartela", "paleta", "numeração", "marcação", "etiqueta", "régua", "suporte"]
ABERTURAS_ETIQUETA = ["o objeto é", "trata-se de"]  # só conta se ABRE o texto
TERMOS_AUSENCIA = ["não há", "sem etiqueta", "sem sinais", "sem evidência", "sem artefatos", "sem marcas"]
TERMOS_FOTO = ["posicionad", "inclinad", "enquadr", "fotografia", "na imagem", "da imagem"]
TERMOS_ESPECULACAO = ["sugere", "sugerindo", "parece ", "parecendo", "possivelmente"]
FRASES_VAZIAS = ["porte médio", "uso prático", "uso frequente", "sinais de uso", "forma funcional", "forma é funcional"]

def tem_atribuicao(texto):
    t = texto.lower()
    return "registro" in t or "catálogo" in t or "catalogo" in t

for obj in objetos:
    p = []
    if not obj["json_valido"]:
        p.append("JSON inválido")
    if not obj["enquadramento_ok"]:
        p.append("observação sem linha ENQUADRAMENTO válida")
    povo = obj["registro"]["Povo"]
    a = obj["alt_text"].lower()
    if povo and povo.split()[0].lower() not in a:
        p.append(f"povo '{povo}' ausente do alt")
    for termo in TERMOS_ARTEFATO:
        if termo in a:
            p.append(f"artefato no alt ('{termo}')")
    if "fundo" in a:
        p.append("'fundo' no alt (sobrou do pós-processamento?)")
    if len(obj["alt_text"].split()) > 30:
        p.append(f"{len(obj['alt_text'].split())} palavras")
    # coerência de enquadramento: "Detalhe" no alt <=> decisão "detalhe" da observação
    alt_detalhe = a.strip().startswith("detalhe")
    if alt_detalhe and obj["enquadramento"] != "detalhe":
        p.append("alt diz 'Detalhe' mas a observação decidiu 'inteiro'")
    if not alt_detalhe and obj["enquadramento"] == "detalhe":
        p.append("observação decidiu 'detalhe' mas o alt não começa com 'Detalhe de'")
    d = obj["descricao_objeto"].lower().strip()
    if obj["descricao_objeto"]:
        if not tem_atribuicao(obj["descricao_objeto"]):
            p.append("nível 2 sem atribuição ao registro")
        if any(d.startswith(ab) for ab in ABERTURAS_ETIQUETA):
            p.append("nível 2 abre com frase-etiqueta")
        if "a função é" in d:
            p.append("frase-etiqueta ('a função é')")
        if "aquisição em" in d:
            p.append("'foi aquisição em' (usar 'adquirido em')")
        for termo in TERMOS_ARTEFATO:
            if termo in d:
                p.append(f"artefato no nível 2 ('{termo}')")
        for termo in TERMOS_AUSENCIA:
            if termo in d:
                p.append(f"afirmação de ausência ('{termo}')")
        for termo in TERMOS_FOTO:
            if termo in d:
                p.append(f"foto no nível 2 ('{termo}')")
    for nome, texto in [("alt", a), ("nível 2", d)]:
        for termo in TERMOS_ESPECULACAO:
            if termo in texto:
                p.append(f"especulação no {nome} ('{termo.strip()}')")
        for termo in FRASES_VAZIAS:
            if termo in texto:
                p.append(f"frase vazia no {nome} ('{termo}')")
    for f in obj["flags"]:
        det = f["detalhe"].lower()
        if f["tipo"] == "artefato_estudio" and "fundo" in det and not any(t in det for t in TERMOS_ARTEFATO):
            p.append("flag de fundo (ruído — fundo de estúdio não é artefato)")
        if any(t in det for t in ["sem ", "não há"]):
            p.append("flag afirmando ausência")
    obj["problemas"] = p
    status = "✓" if not p else "⚠ " + "; ".join(p)
    print(f"{obj['id']} {obj['titulo'][:30]:30} {status}")

abano = next(o for o in objetos if o["id"] == 63283)
abano_ok = abano["enquadramento"] == "detalhe" and abano["alt_text"].strip().lower().startswith("detalhe") and tem_atribuicao(abano["descricao_objeto"])
print(f"\nCaso-referência Abano: {'✓ passou' if abano_ok else '✗ FALHOU'}")
print(f"Total: {sum(1 for o in objetos if not o['problemas'])}/{len(objetos)} objetos sem problemas | "
      f"{sum(len(o['flags']) for o in objetos)} flags ({sum(len(o['artefatos_obs']) for o in objetos)} automáticas de artefato)")

In [ ]:
# Etapa 8 — Salvar no Drive (arquivo v6 — os resultados anteriores ficam preservados)
resultado = {
    "notebook": "04_pipeline_completo_v6",
    "modelo": MODELO,
    "embedding": "Qwen/Qwen3-Embedding-0.6B",
    "rubrica_versao": rubrica["versao"],
    "prompt_observacao_v3": PROMPT_OBSERVACAO_V3,
    "prompt_redacao_v9": PROMPT_REDACAO_V9,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "enquadramento": o["enquadramento"],
         "artefatos_obs": o["artefatos_obs"],
         "alt_text": o["alt_text"], "descricao_objeto": o["descricao_objeto"],
         "flags": o["flags"], "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/04_pipeline_completo_v6.json"
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude que o Notebook 04 **v6** terminou — ele busca o resultado no Drive e roda a comparação decisiva: **v5 (sistema antigo) × v6 (sistema redesenhado), mesmo modelo**. Se os defeitos induzidos sumirem ("sobre a argila bege" fora de cerâmica, "Detalhe" em objeto inteiro, papagaio de exemplos), a hipótese do autor estava certa: o gargalo era o sistema de instruções, não o modelo. Só depois dessa resposta o bake-off de redator volta à mesa.

**O que este notebook prova:** o efeito isolado do redesenho do sistema (observação estruturada + contrato de fontes + garantias em código), medido com o mesmo modelo e os mesmos 20 objetos. **O que ainda não prova:** as métricas nos 40 casos (E8) e a avaliação cega (E10).